## text-specific analysis 

Generating basic info of `.rft` documents separated by month and year. Generated data is in `data.json` file.

In [1]:
import os
from striprtf.striprtf import rtf_to_text
from classla import Pipeline
import classla
import json
import re
import pandas as pd
from datetime import datetime
#import nltk

In [ ]:
#nltk.download("punkt")
classla.download("sl")
nlp = Pipeline("sl", processors="tokenize")
path = "../Data/"
subfolders_year = [f.path for f in os.scandir(path) if f.is_dir()]

In [ ]:
document_dict = {}
date_pattern = r"(\d{1,2})\. (\d{1,2})\. (\d{4})"

for folder_year in subfolders_year:
    folder_year_name = os.path.basename(folder_year)
    subfolders_month = [f.path for f in os.scandir(folder_year) if f.is_dir()]
    document_dict[folder_year_name] = {
        "months": len(subfolders_month),
        "sentence_count": 0,
        "word_count": 0,
        "months_data": {}
    }
    for folder_month in subfolders_month:
        folder_month_name = os.path.basename(folder_month)
        files = [f for f in os.listdir(folder_month)]
        document_dict[folder_year_name]["months_data"][folder_month_name] = {
            "files": len(files),
            "sentence_count": 0,
            "word_count": 0,
            "file_data": {}
        }

        for file in files:
            with open(os.path.join(folder_month, file), "r", encoding="utf-8") as f:
                file_name = os.path.basename(file)
                rtf_content = f.read()
                plain_text = rtf_to_text(rtf_content)
                plain_text = re.sub(r"\s+", " ", plain_text).strip()
                plain_text = plain_text.replace("\u0000", "")
                modified_text = re.sub(date_pattern, r"\1[PERIOD] \2[PERIOD] \3", plain_text)

                # Tokenize plain text into sentences
                #sentences = nltk.tokenize.sent_tokenize(plain_text, language="sl")
                doc = nlp(modified_text)
                sentences = [sentence.text for sentence in doc.sentences]

                sentences = [sentence.replace("[PERIOD]", ".") for sentence in sentences]
                

                document_dict[folder_year_name]["months_data"][folder_month_name]["file_data"][file_name] = {
                    "content": plain_text,
                    "word_count": len(plain_text.split()),
                    "char_count": len(plain_text),
                    "sentence_count": len(sentences),
                    "sentences": sentences
                }

                document_dict[folder_year_name]["months_data"][folder_month_name]["sentence_count"] += len(sentences)
                document_dict[folder_year_name]["months_data"][folder_month_name]["word_count"] += len(plain_text.split())

                document_dict[folder_year_name]["sentence_count"] += len(sentences)
                document_dict[folder_year_name]["word_count"] += len(plain_text.split())
                

with open("data.json", "w", encoding="utf-8") as f:
    json.dump(document_dict, f, ensure_ascii=False, indent=4)


Read RTF documents in `data.json` file to `Pandas DataFrame`.

In [11]:
# Read JSON
with open("data.json", "r", encoding="utf-8") as f:
    document_dict = json.load(f)
data = document_dict

records = []
for year in ["promet_2022", "promet_2023", "promet_2024"]:
    for month in document_dict[year]["months_data"].keys():
        month_data = document_dict[year]["months_data"][month]
        file_data_dict = month_data["file_data"]

        # Create a list of flattened records
        for file_name, file_details in file_data_dict.items():
            record = {
                # File-specific fields
                "year": year,
                "month": month,
                "file_name": file_name,
                "content": file_details["content"],
                "file_word_count": file_details["word_count"],
                "file_char_count": file_details["char_count"],
                "file_sentence_count": file_details["sentence_count"],
                "sentences": file_details["sentences"],
                
                # Month-specific fields
                "month_files": month_data["files"],
                "month_sentence_count": month_data["sentence_count"],
                "month_word_count": month_data["word_count"],

                # Year-specific fields
                "months": document_dict[year]["months"],
                "year_sentence_count": document_dict[year]["sentence_count"],
                "year_word_count": document_dict[year]["word_count"]
            }

            # Add datetime parsing
            content_text = file_details["content"]
            match = re.search(r"Prometne informacije\s+(\d{1,2}\.\s*\d{1,2}\.\s*\d{4})\s+(\d{1,2}\.\d{2})", content_text)
            if match:
                date_part = match.group(1).replace(" ", "") 
                time_part = match.group(2)                
                datetime_str = f"{date_part} {time_part}"
                try:
                    parsed_datetime = datetime.strptime(datetime_str, "%d.%m.%Y %H.%M")
                except ValueError:
                    parsed_datetime = None
            else:
                parsed_datetime = None
            record["datetime"] = parsed_datetime

            records.append(record)

# Convert to DataFrame
RTFS = pd.DataFrame(records)
RTFS.head()

,year,month,file_name,content,file_word_count,file_char_count,file_sentence_count,sentences,month_files,month_sentence_count,month_word_count,months,year_sentence_count,year_word_count,datetime
0,promet_2022,april_2022,TMP-1.rtf,Prometne informacije 30. 04. 2022 18.30 1. in ...,74,431,4,[Prometne informacije 30. 04. 2022 18.30 1. in...,741,3832,55464,12,51490,740176,2022-04-30 18:30:00
1,promet_2022,april_2022,TMP-10.rtf,Prometne informacije 30. 04. 2022 13.00 1. in ...,61,380,3,[Prometne informacije 30. 04. 2022 13.00 1. in...,741,3832,55464,12,51490,740176,2022-04-30 13:00:00
2,promet_2022,april_2022,TMP-100.rtf,Prometne informacije 27. 04. 2022 6.30 1. prog...,47,337,3,[Prometne informacije 27. 04. 2022 6.30 1. pro...,741,3832,55464,12,51490,740176,2022-04-27 06:30:00
3,promet_2022,april_2022,TMP-101.rtf,Prometne informacije 27. 04. 2022 6.00 1. in 2...,47,294,3,[Prometne informacije 27. 04. 2022 6.00 1. in ...,741,3832,55464,12,51490,740176,2022-04-27 06:00:00
4,promet_2022,april_2022,TMP-102.rtf,Prometne informacije 26. 04. 2022 20.00 2. pro...,71,467,5,[Prometne informacije 26. 04. 2022 20.00 2. pr...,741,3832,55464,12,51490,740176,2022-04-26 20:00:00


Link each report from `traffic_reports_2022_2023_2024.xlsx` to corresponding RTF document.

In [12]:
# Read the "traffic_reports_2022_2023_2024.xlsx" to Pandas DataFrame

path = "../Data/traffic_reports_2022_2023_2024.xlsx"
years = [2022, 2023, 2024]
REPORTS = {}

def clean_cell(val):
    if isinstance(val, str):
        val = re.sub(r'<strong>.*?</strong>', '', val, flags=re.DOTALL)  # removes <strong>...</strong>
        val = re.sub(r'</?p>', '', val) # removes <p> and </p>
        val = val.strip() # removes spaces             
    return val

for year in years:
    REPORTS[year] = pd.read_excel(path, sheet_name=str(year))

    "Some data cleaning and preprocessing"

    # Drop 100% empty columns A2 and C1
    REPORTS[year].drop(columns=["A2", "C1"], inplace=True)

    # Drop English columns
    REPORTS[year].drop(columns=["B2", "C2"], inplace=True)

    # Drop useless columns (report's id, report's author, all titles)
    REPORTS[year].drop(columns=["LegacyId", "Operater", "TitlePomembnoSLO", "TitleNesreceSLO", "TitleZastojiSLO", "TitleVremeSLO", "TitleOvireSLO","TitleDeloNaCestiSLO", "TitleOpozorilaSLO", "TitleMednarodneInformacijeSLO", "TitleSplosnoSLO"], inplace=True)

    # Clean up text values
    REPORTS[year] = REPORTS[year].map(clean_cell)

REPORTS[2022].head()

,Datum,A1,B1,ContentPomembnoSLO,ContentNesreceSLO,ContentZastojiSLO,ContentVremeSLO,ContentOvireSLO,ContentDeloNaCestiSLO,ContentOpozorilaSLO,ContentMednarodneInformacijeSLO,ContentSplosnoSLO
0,2022-01-01 00:07:07,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
1,2022-01-01 00:07:29,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
2,2022-01-01 00:07:30,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
3,2022-01-01 00:07:36,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN
4,2022-01-01 00:16:26,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,Na primorski avtocesti je ponovno odprt priklj...,Po Sloveniji velja med prazniki omejitev za to...,NaN,NaN


In [ ]:
# For each report, find corresponding RTF file

RTFS_sorted = RTFS.sort_values("datetime").reset_index(drop=True)
for year in years:
    REPORTS[year]["RTF_file_name"] = None  # New column for RTF file names
    
    for index, row in REPORTS[year].iterrows():
        report_date = datetime.strptime(str(row["Datum"]), "%Y-%m-%d %H:%M:%S")

        # Find first RTF document with its datetime after the current report's datetime
        match = RTFS_sorted[RTFS_sorted["datetime"] >= report_date]
        
        if not match.empty:
            matched_file = match.iloc[0]["file_name"]  # Get the first match
        else:
            matched_file = None  # No match found

        REPORTS[year].at[index, "RTF_file_name"] = matched_file

In [ ]:
# Save new reports data to one CSV file

all_reports = pd.concat([REPORTS[year] for year in years], ignore_index=True)
all_reports.to_csv("traffic_reports_linked.csv", index=False, encoding='utf-8')

In [ ]:
# There are 5 reports without RTF file (last 5 reports from year 2024) due to no RTF data from 2025

REPORTS_linked = pd.read_csv("traffic_reports_linked.csv", encoding='utf-8')
REPORTS_linked.loc[REPORTS_linked['RTF_file_name'].isnull()]

,Datum,A1,B1,ContentPomembnoSLO,ContentNesreceSLO,ContentZastojiSLO,ContentVremeSLO,ContentOvireSLO,ContentDeloNaCestiSLO,ContentOpozorilaSLO,ContentMednarodneInformacijeSLO,ContentSplosnoSLO,RTF_file_name
170368,2024-12-31 21:09:30,NaN,"Omejitev za tovorna vozila, katerih največja d...",NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,"<a href=""https://www.promet.si/sl/aktualna-nap...",Srečno in varno na poti v letu 2025.,NaN,"Omejitev za tovorna vozila, katerih največja d...",NaN
170369,2024-12-31 21:53:27,NaN,"Na A5, Maribor - Pince, pred priključkom Lenar...",NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,"Na A5, Maribor - Pince, pred priključkom Lenar...","<a href=""https://www.promet.si/sl/aktualna-nap...",Srečno in varno na poti v letu 2025.,NaN,"Omejitev za tovorna vozila, katerih največja d...",NaN
170370,2024-12-31 22:08:45,NaN,"Omejitev za tovorna vozila, katerih največja d...",NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,"<a href=""https://www.promet.si/sl/aktualna-nap...",Srečno in varno na poti v letu 2025.,NaN,"Omejitev za tovorna vozila, katerih največja d...",NaN
170371,2024-12-31 22:55:49,NaN,"Omejitev za tovorna vozila, katerih največja d...",NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,"<a href=""https://www.promet.si/sl/aktualna-nap...",Srečno in varno na poti v letu 2025.,NaN,"Omejitev za tovorna vozila, katerih največja d...",NaN
170372,2024-12-31 23:29:15,NaN,"Omejitev za tovorna vozila, katerih največja d...",NaN,NaN,NaN,Ponekod po Sloveniji megla v pasovih zmanjšuje...,NaN,"<a href=""https://www.promet.si/sl/aktualna-nap...",Srečno in varno na poti v letu 2025.,NaN,"Omejitev za tovorna vozila, katerih največja d...",NaN
